# wmh-multisite — 2e entraînement nnU-Net : augmentation forte DA5 (Kaggle, 2 × T4)

Identique au notebook `03_nnunet_train` (mêmes données, même architecture, même fold, 250 epochs), à une seule différence : le trainer `nnUNetTrainerWMH_DA5_250` (augmentation de données plus agressive). But : mesurer l'effet de l'augmentation sur la robustesse aux scanners inconnus, à budget égal (J-037).

Étape 5.2 de la ROADMAP. Entraîne nnU-Net v2 sur `Dataset001_WMH` (FLAIR + T1 corrigées par N4 avec le masque HD-BET, T1 recalée sur la grille FLAIR), fold 0 de nos folds stratifiés par site et par charge lésionnelle (J-032).

**Réglages (J-037)**

| Réglage | Valeur | Raison |
|---|---|---|
| Architecture | **ResEnc M** (`nnUNetPlannerResEncM`) | recommandée par « nnU-Net Revisited » (2024) ; cible 8 Go de mémoire GPU (T4 : 16 Go) |
| Configuration | `3d_fullres` | |
| Epochs | **250** | durée réglée sur ce qu'on termine : le taux d'apprentissage décroît jusqu'à ~0 à la dernière epoch |
| Trainer | `nnUNetTrainerWMH_250` (standard) ou `nnUNetTrainerWMH_DA5_250` (augmentation forte, 2e entraînement) | même budget, seule l'augmentation change |
| Perte | Dice + entropie croisée (défaut nnU-Net) | la plus classique |
| Sur-échantillonnage des lésions | 33 % des patches (défaut) | |
| GPU | 2 × T4 en parallèle (DDP), `torch.compile`, 4 processus d'augmentation | benchmark J-019 : ÷ 1,96 sur le temps |
| Sauvegarde | `checkpoint_latest.pth` toutes les **50 epochs** | reprise possible sans perdre plus de 50 epochs |

**Sessions et reprise.** Une session Kaggle dure au plus 12 h. Le notebook surveille l'entraînement et, si la prochaine sauvegarde ne peut plus être atteinte avant la limite (`SESSION_BUDGET_H`), il **s'arrête proprement juste après une sauvegarde**. Tout est écrit dans `/kaggle/working`, donc conservé dans la sortie de la version.

## Instructions

**Première session**
1. *Add Input* : le dataset privé contenant `Dataset001_WMH` (**nouvelle version**, déposée après le N4 HD-BET).
2. *Settings* : Accelerator **GPU T4 x2**, Internet **On**.
3. Vérifier `TRAINER` dans la cellule de configuration.
4. *Save Version* > **Save & Run All (Commit)** : le calcul tourne sans navigateur ouvert.

**Si la session s'arrête avant la fin** (message « session budget reached ») : créer une nouvelle version du même notebook, *Add Input* > la **sortie de la version précédente** (onglet *Notebooks* > *Your Work* > ce notebook), puis Save & Run All. Le notebook trouve le dernier `checkpoint_latest.pth` et reprend (`--c`).

**À la fin** : copier dans la conversation le bloc JSON de la dernière cellule ; la sortie contient le modèle, les courbes (`progress.png`), les métriques par epoch (`metrics.csv`) et la validation sur les 12 cas du fold 0.

In [ ]:
# ---- Configuration ----
TRAINER = "nnUNetTrainerWMH_DA5_250"      # 2nd training: strong augmentation (J-037)
PLANNER, PLANS = "nnUNetPlannerResEncM", "nnUNetResEncUNetMPlans"
CONFIG, FOLD, DATASET = "3d_fullres", 0, "Dataset001_WMH"
N_PROC_DA = 4
SESSION_BUDGET_H = 11.3                  # stop after a checkpoint if the next one would exceed this
VALIDATION_MARGIN_MIN = 30               # time kept for the final validation of the 12 cases

In [ ]:
# 1. Environment
import glob, json, os, re, shutil, subprocess, time, zipfile
from pathlib import Path
import psutil, torch

T_START = time.time()
REPORT = {"started": time.strftime("%Y-%m-%d %H:%M:%S"), "trainer": TRAINER, "plans": PLANS,
          "env": {"torch": torch.__version__, "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                  "cpus": os.cpu_count(), "ram_gb": round(psutil.virtual_memory().total / 1e9, 1)}}
NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS >= 1, "No GPU: set Accelerator to GPU T4 x2"
print(REPORT["env"])
!pip install -q nnunetv2==2.8.1 2>&1 | tail -2
REPORT["env"]["nnunetv2"] = "2.8.1"

In [ ]:
# 2. Custom trainers: 250 epochs, checkpoint every 50 epochs (standard and strong augmentation DA5)
import nnunetv2
trainer_file = Path(nnunetv2.__file__).parent / "training/nnUNetTrainer/variants/wmh_multisite_trainers.py"
trainer_file.write_text('''
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.training.nnUNetTrainer.variants.data_augmentation.nnUNetTrainerDA5 import nnUNetTrainerDA5


class nnUNetTrainerWMH_250(nnUNetTrainer):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = 250
        self.save_every = 50


class nnUNetTrainerWMH_DA5_250(nnUNetTrainerDA5):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = 250
        self.save_every = 50
''')
print("trainers written to", trainer_file)

In [ ]:
# 3. Dataset: Kaggle decompresses the .nii.gz of an uploaded archive -> use the real file ending
found = glob.glob("/kaggle/input/**/Dataset001_WMH/dataset.json", recursive=True)
if not found:
    zips = glob.glob("/kaggle/input/**/Dataset001_WMH.zip", recursive=True)
    assert zips, "Dataset001_WMH not found in /kaggle/input: add the private dataset with 'Add Input'"
    zipfile.ZipFile(zips[0]).extractall("/tmp/raw_zip")
    found = glob.glob("/tmp/raw_zip/**/Dataset001_WMH/dataset.json", recursive=True)
SRC = Path(found[0]).parent
counts = {f: sorted(p.name for p in (SRC / f).iterdir()) for f in ("imagesTr", "labelsTr")}
ending = ".nii.gz" if counts["labelsTr"][0].endswith(".nii.gz") else ".nii"
assert sum(n.endswith(ending) for n in counts["imagesTr"]) == 120 and sum(n.endswith(ending) for n in counts["labelsTr"]) == 60

RAW = Path("/tmp/raw/Dataset001_WMH")                    # writable copy made of symbolic links
shutil.rmtree(RAW, ignore_errors=True)
for folder in ("imagesTr", "labelsTr"):
    (RAW / folder).mkdir(parents=True)
    for f in (SRC / folder).iterdir():
        if f.name.endswith(ending):
            (RAW / folder / f.name).symlink_to(f)
ds = json.load(open(SRC / "dataset.json")); ds["file_ending"] = ending
json.dump(ds, open(RAW / "dataset.json", "w"), indent=2)
REPORT["dataset"] = {"source": str(SRC), "file_ending": ending, "description": ds.get("description", "")}
print(REPORT["dataset"])

In [ ]:
# 4. nnU-Net folders; results in /kaggle/working so that they are kept with the notebook output.
#    Resume: if a previous session's output is attached as input, copy its results here.
os.environ["nnUNet_raw"] = "/tmp/raw"
os.environ["nnUNet_preprocessed"] = "/tmp/preprocessed"
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"
OUT = Path(os.environ["nnUNet_results"]) / DATASET / f"{TRAINER}__{PLANS}__{CONFIG}" / f"fold_{FOLD}"
previous = sorted(glob.glob(f"/kaggle/input/**/nnUNet_results/{DATASET}/{TRAINER}__{PLANS}__{CONFIG}/fold_{FOLD}/checkpoint_*.pth", recursive=True))
if previous and not OUT.exists():
    src_results = Path(previous[0]).parents[3]          # .../nnUNet_results
    shutil.copytree(src_results, os.environ["nnUNet_results"], dirs_exist_ok=True)
    print("resuming from", src_results)
DONE = (OUT / "checkpoint_final.pth").is_file()
RESUME = (OUT / "checkpoint_latest.pth").is_file() and not DONE
REPORT["session"] = {"resume": RESUME, "already_done": DONE}
print(REPORT["session"])

In [ ]:
# 5. Planning and preprocessing (ResEnc M), then our stratified folds (J-032)
t0 = time.time()
r = subprocess.run(f"nnUNetv2_plan_and_preprocess -d 1 -c {CONFIG} -pl {PLANNER} -np 4 --verify_dataset_integrity",
                   shell=True, capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-1200:])
assert r.returncode == 0, "planning/preprocessing failed"
PRE = Path(os.environ["nnUNet_preprocessed"]) / DATASET
shutil.copy(SRC / "splits_final.json", PRE / "splits_final.json")
plan = json.load(open(PRE / f"{PLANS}.json"))["configurations"][CONFIG]
REPORT["plan"] = {k: plan.get(k) for k in ("patch_size", "batch_size", "spacing")}
REPORT["preprocess_s"] = round(time.time() - t0, 1)
print(REPORT["plan"], REPORT["preprocess_s"], "s")

In [ ]:
# 6. Training, watched: stop cleanly after a checkpoint if the next one cannot be reached in time
def kill_tree(pid):
    try:
        parent = psutil.Process(pid)
        for p in parent.children(recursive=True) + [parent]:
            p.kill()
    except psutil.Error:
        pass

def epoch_times():
    logs = sorted(OUT.glob("training_log_*.txt"))
    return [float(x) for log in logs for x in re.findall(r"Epoch time: ([\d.]+) s", log.read_text())]

stopped = None
if not DONE:
    env = dict(os.environ, nnUNet_n_proc_DA=str(N_PROC_DA), nnUNet_compile="t")
    cmd = f"nnUNetv2_train 1 {CONFIG} {FOLD} -tr {TRAINER} -p {PLANS} -num_gpus {NUM_GPUS}" + (" --c" if RESUME else "")
    print(cmd)
    proc = subprocess.Popen(cmd, shell=True, env=env, stdout=open("/kaggle/working/train_stdout.log", "a"),
                            stderr=subprocess.STDOUT)
    budget_s = SESSION_BUDGET_H * 3600
    last_ckpt = (OUT / "checkpoint_latest.pth").stat().st_mtime if (OUT / "checkpoint_latest.pth").exists() else None
    last_status = 0
    while proc.poll() is None:
        time.sleep(30)
        if time.time() - last_status > 600:                                # progress line every 10 min
            last_status = time.time()
            times = epoch_times()
            if times:
                per = sorted(times[-10:])[len(times[-10:]) // 2]
                left_h = (250 - len(times)) * per / 3600
                logs = sorted(OUT.glob("training_log_*.txt"))
                dice = re.findall(r"Pseudo dice \[(?:np\.float32\()?([\d.]+)", logs[-1].read_text()) if logs else []
                print(f"[{(time.time() - T_START) / 3600:.2f} h] epochs done: {len(times)}/250 | {per:.0f} s/epoch"
                      f" | training left ~{left_h:.1f} h | last pseudo Dice {dice[-1] if dice else '-'}", flush=True)
        ckpt = OUT / "checkpoint_latest.pth"
        if ckpt.exists() and ckpt.stat().st_mtime != last_ckpt:          # a checkpoint was just written
            time.sleep(60)                                                 # let the write finish
            last_ckpt = ckpt.stat().st_mtime
            times = epoch_times()
            per_epoch = sorted(times[-10:])[len(times[-10:]) // 2] if times else 0
            elapsed = time.time() - T_START
            needed = 50 * per_epoch + VALIDATION_MARGIN_MIN * 60
            print(f"checkpoint at epoch {len(times)} | {per_epoch:.0f} s/epoch | elapsed {elapsed/3600:.2f} h")
            if elapsed + needed > budget_s:
                stopped = f"session budget reached after epoch {len(times)}: resume in a new session"
                kill_tree(proc.pid)
                break
    REPORT["returncode"] = proc.wait()
REPORT["stopped"] = stopped
print(stopped or "training process finished")
print(open("/kaggle/working/train_stdout.log").read()[-2500:])

In [ ]:
# 7. Metrics per epoch (from nnU-Net's training logs), curves and final validation
rows, cur = [], {}
for log in sorted(OUT.glob("training_log_*.txt")):
    for line in log.read_text().splitlines():
        m = re.search(r":\s*Epoch (\d+)\s*$", line)
        if m:
            if cur: rows.append(cur)
            cur = {"epoch": int(m[1])}
        for key, pat in [("lr", r"Current learning rate: ([\d.e-]+)"), ("train_loss", r"train_loss (-?[\d.]+)"),
                         ("val_loss", r"val_loss (-?[\d.]+)"), ("pseudo_dice", r"Pseudo dice \[(?:np\.float32\()?([\d.]+)"),
                         ("epoch_s", r"Epoch time: ([\d.]+) s")]:
            m = re.search(pat, line)
            if m and cur: cur[key] = float(m[1])
if cur: rows.append(cur)
import pandas as pd
metrics = pd.DataFrame(rows)
if "epoch" in metrics:
    metrics = metrics.drop_duplicates("epoch", keep="last").sort_values("epoch")
else:
    print("WARNING: no epoch header parsed in the training logs; metrics.csv will be empty")
    metrics = pd.DataFrame(columns=["epoch", "lr", "train_loss", "val_loss", "pseudo_dice", "epoch_s"])
metrics.to_csv("/kaggle/working/metrics.csv", index=False)
print(metrics.tail(5).to_string(index=False))

summary_file = OUT / "validation" / "summary.json"
validation = None
if summary_file.is_file():
    s = json.load(open(summary_file))
    validation = {"foreground_mean_dice": s["foreground_mean"]["Dice"],
                  "per_case_dice": {c["prediction_file"].split("/")[-1]: c["metrics"]["1"]["Dice"] for c in s["metric_per_case"]}}
REPORT["epochs_done"] = int(metrics.epoch.max() + 1) if len(metrics) else 0
REPORT["s_per_epoch_median"] = float(metrics.epoch_s.median()) if "epoch_s" in metrics else None
REPORT["last"] = metrics.iloc[-1].to_dict() if len(metrics) else None
REPORT["best_pseudo_dice"] = float(metrics.pseudo_dice.max()) if "pseudo_dice" in metrics else None
REPORT["validation_fold0"] = validation
REPORT["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
REPORT["session_h"] = round((time.time() - T_START) / 3600, 2)
json.dump(REPORT, open("/kaggle/working/train_report.json", "w"), indent=1)
print("\n=== Copier ce bloc dans la conversation ===")
print(json.dumps({k: v for k, v in REPORT.items() if k != "validation_fold0"} | {"validation_mean_dice": validation and validation["foreground_mean_dice"]}, indent=1, default=str))